In [ ]:
import random
import os
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D, Dense, Input

def build_conv1d_tcn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(filters=64, kernel_size=3, padding='causal', activation='relu', dilation_rate=1),
        Conv1D(filters=64, kernel_size=3, padding='causal', activation='relu', dilation_rate=2),
        Conv1D(filters=32, kernel_size=3, padding='causal', activation='relu', dilation_rate=4),
        GlobalAveragePooling1D(),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# Setup
data_dir = "../windowed_data/6h"
feature_names = ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
selected_features = ['bg', 'steps', 'dist', 'insulin']
selected_indices = [feature_names.index(f) for f in selected_features]
all_files = [f for f in os.listdir(data_dir) if f.endswith('.npz')]

results_tcn = {}
tcn_preds = {}

# Train & Evaluate per Patient
for file in all_files:
    pid = file.replace(".npz", "")
    data = np.load(os.path.join(data_dir, file))
    X, y = data['X'], data['y']
    X = X[:, :, selected_indices]

    # Chronological 80/20 split
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    print(f"\nTraining TCN for {pid}...")
    model = build_conv1d_tcn_model(input_shape=(X.shape[1], X.shape[2]))
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

    y_pred = model.predict(X_test).flatten()
    rmse_tcn = np.sqrt(mean_squared_error(y_test, y_pred))
    mae_tcn = mean_absolute_error(y_test, y_pred)

    results_tcn[pid] = {"rmse": rmse_tcn, "mae": mae_tcn}
    tcn_preds[pid] = {"y_true": y_test.tolist(), "y_pred": y_pred.tolist()}

    print(f"{pid} - RMSE: {rmse_tcn:.3f} | MAE: {mae_tcn:.3f}")


Training TCN for P01...
Epoch 1/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 48s 28ms/step - loss: 20.2815
Epoch 2/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 55s 38ms/step - loss: 10.5158
Epoch 3/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - loss: 8.4202
Epoch 4/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 35s 24ms/step - loss: 7.4896
Epoch 5/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 6.2728
Epoch 6/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - loss: 5.7281
Epoch 7/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 40s 28ms/step - loss: 5.1690
Epoch 8/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 25s 17ms/step - loss: 4.8261
Epoch 9/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 36s 25ms/step - loss: 4.4680
Epoch 10/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - loss: 4.3168
365/365 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
P01 - RMSE: 3.346 | MAE: 2.498

Training TCN for P02...
Epoch 1/10
1506/1506 ━━━━━━━━━━━━━━━━━━━━ 22s 13ms/step - loss: 95.8216
Epoch 2/10
1506/1506 ━━━━━━━━━━━━━━━━━━━━ 26s 17ms/step - loss: 71.4131
Epoch 3/10
1506/1506 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Save results
df_results_tcn = pd.DataFrame(results_tcn).T.round(3)
df_results_tcn.index.name = "Patient ID"

# Results visualisation
# RMSE Plot
df_rmse_sorted_tcn = df_results_tcn.sort_values("rmse")
x_rmse_tcn = np.arange(len(df_rmse_sorted_tcn))
rmse_values_tcn = df_rmse_sorted_tcn["rmse"].values
mean_rmse_tcn = rmse_values_tcn.mean()
std_rmse_tcn = rmse_values_tcn.std()

plt.figure(figsize=(12, 5))
plt.bar(x_rmse_tcn, rmse_values_tcn, color='skyblue')
plt.xticks(x_rmse_tcn, df_rmse_sorted_tcn.index, rotation=45)

# Confidence band 
plt.axhspan(mean_rmse_tcn - std_rmse_tcn, mean_rmse_tcn + std_rmse_tcn, color='red', alpha=0.2, label="±1 SD")
plt.axhline(mean_rmse_tcn, color='red', linestyle='--', label=f"Mean RMSE = {mean_rmse_tcn:.2f}")

plt.title("RMSE per Patient - TCN")
plt.ylabel("RMSE")
plt.legend()
plt.tight_layout()
plt.show()


# MAE Plot
df_mae_sorted_tcn = df_results_tcn.sort_values("mae")
x_mae_tcn = np.arange(len(df_mae_sorted_tcn))
mae_values_tcn = df_mae_sorted_tcn["mae"].values
mean_mae_tcn = mae_values_tcn.mean()
std_mae_tcn = mae_values_tcn.std()

plt.figure(figsize=(12, 5))
plt.bar(x_mae_tcn, mae_values_tcn, color='skyblue')
plt.xticks(x_mae_tcn, df_mae_sorted_tcn.index, rotation=45)

# Confidence band 
plt.axhspan(mean_mae_tcn - std_mae_tcn, mean_mae_tcn + std_mae_tcn, color='green', alpha=0.2, label="±1 SD")
plt.axhline(mean_mae_tcn, color='green', linestyle='--', label=f"Mean MAE = {mean_mae_tcn:.2f}")

plt.title("MAE per Patient - TCN")
plt.ylabel("MAE")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_results_tcn.to_csv("../output_baseline/tcn_per_patient_results.csv")

with open("../output_baseline/tcn_predictions_per_patient.json", "w") as f:
    json.dump(tcn_preds, f)

print("\nAverage RMSE:", df_results_tcn["rmse"].mean())
print("Average MAE:", df_results_tcn["mae"].mean())
display(df_results_tcn)

In [ ]:
# Load TCN predictions
with open("../output_baseline/tcn_predictions_per_patient.json", "r") as f:
    tcn_preds = json.load(f)

# Convert to DataFrame 
rows = []
for pid, data in tcn_preds.items():
    for true_val, pred_val in zip(data["y_true"], data["y_pred"]):
        rows.append({
            "patient_id": pid,
            "true_bg": float(true_val),
            "predicted_bg": float(pred_val)
        })

df_tcn = pd.DataFrame(rows)

# Clarke Error Grid Zone Classification (using mg/dL)
def classify_ceg_zone(true, pred):
    true = float(true)
    pred = float(pred)

    # Zone A: within 20% of reference BG or both < 70
    if (abs(true - pred) <= 20) or (true < 70 and pred < 70) or (true > 180 and pred > 180 and abs(true - pred) <= 20):
        return 'A'
    
    # Zone B: outside 20% but no treatment error (e.g. still within euglycemic range)
    if (true >= 70 and true <= 180 and pred >= 70 and pred <= 180):
        return 'B'
    if (true < 70 and 70 <= pred <= 180):
        return 'B'
    if (true > 180 and 70 <= pred <= 180):
        return 'B'
    
    # Zone E: dangerous confusion
    if (true < 70 and pred > 180) or (true > 180 and pred < 70):
        return 'E'
    
    # Zone D: dangerous failure to detect and treat
    if (true >= 180 and pred <= 180) or (true < 70 and pred >= 70):
        return 'D'
    
    # Zone C: overcorrection / unnecessary treatment
    return 'C'

# Convert mmol/L to mg/dL for Clarke Error Grid
df_tcn["true_bg_mg"] = df_tcn["true_bg"] * 18
df_tcn["predicted_bg_mg"] = df_tcn["predicted_bg"] * 18

# Apply classification
df_tcn["zone"] = df_tcn.apply(lambda row: classify_ceg_zone(row["true_bg_mg"], row["predicted_bg_mg"]), axis=1)

# Per-patient zone summary
ceg_summary_tcn = df_tcn.groupby(["patient_id", "zone"]).size().unstack(fill_value=0)
ceg_summary_tcn["Total"] = ceg_summary_tcn.sum(axis=1)
ceg_summary_percent_tcn = ceg_summary_tcn.div(ceg_summary_tcn["Total"], axis=0) * 100
ceg_summary_percent_tcn = ceg_summary_percent_tcn.round(2)
ceg_summary_percent_tcn.to_csv("../output_baseline/tcn_ceg_zone_summary_per_patient.csv")

print("\nTCN Clarke Error Grid Summary (%):")
display(ceg_summary_percent_tcn)

In [ ]:
# Patient-Level Clarke Error Grid Bar Plot
ceg_plot_tcn = ceg_summary_percent_tcn.drop(columns="Total").fillna(0)

plt.figure(figsize=(12, 6))
ceg_plot_tcn.plot(kind="bar", stacked=True, colormap="tab20", figsize=(14, 6))
plt.ylabel("Percentage (%)")
plt.title("Clarke Error Grid Zone Distribution per Patient (TCN)")
plt.legend(title="Zone", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(axis='y')
plt.show()